In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm
import random
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [2]:
# We will use state data instead of regional data 
DATA_DIR = Path.cwd() / "data"
DATA_DIR

PosixPath('/home/zhanglab/tomas/nyu/tr_gy_6113/project/data')

## Load in employment data

In [3]:
cbp_df = pd.read_csv(DATA_DIR / "cbp22st.txt", dtype={"fipstate": str, "naics": str})
cbp_df = cbp_df[cbp_df["lfo"] == "-"]
cbp_df["code"] = cbp_df["naics"].str.rstrip("/-")          # "311///" -> "311"
emp_df = cbp_df.pivot_table(index="fipstate", columns="code", values="emp", aggfunc="sum").fillna(0)
emp_df

code,,11,113,1131,11311,113110,1132,11321,113210,1133,...,813910,81392,813920,81393,813930,81394,813940,81399,813990,99
fipstate,,,,,,,,,,,,,,,,,,,,,
01,1777657.0,5501.0,3785.0,259.0,259.0,259.0,43.0,43.0,43.0,3483.0,...,1447.0,351.0,351.0,1025.0,1025.0,48.0,48.0,344.0,344.0,139.0
02,257462.0,675.0,157.0,0.0,0.0,0.0,0.0,0.0,0.0,150.0,...,226.0,157.0,157.0,308.0,308.0,19.0,19.0,92.0,92.0,14.0
04,2787701.0,1941.0,122.0,0.0,0.0,0.0,0.0,0.0,0.0,73.0,...,1083.0,1265.0,1265.0,899.0,899.0,308.0,308.0,4222.0,4222.0,167.0
05,1089044.0,4780.0,2200.0,247.0,247.0,247.0,23.0,23.0,23.0,1930.0,...,797.0,166.0,166.0,430.0,430.0,41.0,41.0,838.0,838.0,97.0
06,16032440.0,33361.0,2126.0,276.0,276.0,276.0,102.0,102.0,102.0,1748.0,...,8336.0,5136.0,5136.0,15018.0,15018.0,2109.0,2109.0,12803.0,12803.0,1577.0
08,2481196.0,1754.0,104.0,26.0,26.0,26.0,0.0,0.0,0.0,78.0,...,1737.0,1799.0,1799.0,1439.0,1439.0,198.0,198.0,3388.0,3388.0,195.0
09,1508634.0,325.0,33.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,...,1721.0,640.0,640.0,2028.0,2028.0,74.0,74.0,779.0,779.0,75.0
10,424303.0,136.0,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,340.0,87.0,87.0,339.0,339.0,31.0,31.0,131.0,131.0,208.0
11,527355.0,15.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,11796.0,9919.0,9919.0,7039.0,7039.0,2602.0,2602.0,865.0,865.0,13.0


In [4]:
# codes come from: https://www2.census.gov/programs-surveys/cbp/technical-documentation/reference/naics-descriptions/naics2017.txt

sector_map = {"emp_agriculture": ["11"],                    # agriculture, forestry, fishing, and hunting
              "emp_manufacturing": ["31"],                  # manufacturing (31-33)
              "emp_mining": ["21"],                         # mining, oil and gas extraction
              "emp_wholesale_warehousing": ["42", "493"]}   # wholesale trade + warehousing and storage
E_prod = pd.DataFrame({col: emp_df[codes].sum(axis=1) for col, codes in sector_map.items()})
E_prod

,emp_agriculture,emp_manufacturing,emp_mining,emp_wholesale_warehousing
fipstate,,,,
01,5501.0,267658.0,4902.0,94834.0
02,675.0,10824.0,9214.0,9025.0
04,1941.0,171133.0,12105.0,129973.0
05,4780.0,161239.0,3076.0,60097.0
06,33361.0,1160856.0,15982.0,989048.0
08,1754.0,135443.0,19012.0,119528.0
09,325.0,151851.0,527.0,77856.0
10,136.0,29767.0,271.0,26782.0
11,15.0,1014.0,0.0,3433.0


## Load metadata about states, i.e., FIPS codes and commodity codes

In [5]:
metadata = pd.read_excel(DATA_DIR / "state" / "FAF6_Metadata.xlsx", ["State", "Commodity (SCTG2)"])

In [6]:
states = metadata["State"]
state_to_code = dict(zip(states["Description"], states["Numeric Label"]))
state_to_code

{'Alabama': 1,
 'Alaska': 2,
 'Arizona': 4,
 'Arkansas': 5,
 'California': 6,
 'Colorado': 8,
 'Connecticut': 9,
 'Delaware': 10,
 'Washington DC': 11,
 'Florida': 12,
 'Georgia': 13,
 'Hawaii': 15,
 'Idaho': 16,
 'Illinois': 17,
 'Indiana': 18,
 'Iowa': 19,
 'Kansas': 20,
 'Kentucky': 21,
 'Louisiana': 22,
 'Maine': 23,
 'Maryland': 24,
 'Massachusetts': 25,
 'Michigan': 26,
 'Minnesota': 27,
 'Mississippi': 28,
 'Missouri': 29,
 'Montana': 30,
 'Nebraska': 31,
 'Nevada': 32,
 'New Hampshire': 33,
 'New Jersey': 34,
 'New Mexico': 35,
 'New York': 36,
 'North Carolina': 37,
 'North Dakota': 38,
 'Ohio': 39,
 'Oklahoma': 40,
 'Oregon': 41,
 'Pennsylvania': 42,
 'Rhode Island': 44,
 'South Carolina': 45,
 'South Dakota': 46,
 'Tennessee': 47,
 'Texas': 48,
 'Utah': 49,
 'Vermont': 50,
 'Virginia': 51,
 'Washington': 53,
 'West Virginia': 54,
 'Wisconsin': 55,
 'Wyoming': 56}

In [7]:
commodities = metadata["Commodity (SCTG2)"]
code_to_commodity = dict(zip(commodities["Numeric Label"], commodities["Description"]))
code_to_commodity

{1: 'Live animals/fish',
 2: 'Cereal grains',
 3: 'Other ag prods.',
 4: 'Animal feed',
 5: 'Meat/seafood',
 6: 'Milled grain prods.',
 7: 'Other foodstuffs',
 8: 'Alcoholic beverages',
 9: 'Tobacco prods.',
 10: 'Building stone',
 11: 'Natural sands',
 12: 'Gravel',
 13: 'Nonmetallic minerals',
 14: 'Metallic ores',
 15: 'Coal',
 16: 'Crude petroleum',
 17: 'Gasoline',
 18: 'Fuel oils',
 19: 'Natural gas and other fossil products',
 20: 'Basic chemicals',
 21: 'Pharmaceuticals',
 22: 'Fertilizers',
 23: 'Chemical prods.',
 24: 'Plastics/rubber',
 25: 'Logs',
 26: 'Wood prods.',
 27: 'Newsprint/paper',
 28: 'Paper articles',
 29: 'Printed prods.',
 30: 'Textiles/leather',
 31: 'Nonmetal min. prods.',
 32: 'Base metals',
 33: 'Articles-base metal',
 34: 'Machinery',
 35: 'Electronics',
 36: 'Motorized vehicles',
 37: 'Transport equip.',
 38: 'Precision instruments',
 39: 'Furniture',
 40: 'Misc. mfg. prods.',
 41: 'Waste/scrap',
 43: 'Mixed freight'}

## Load in GDP and per capita personal income values

In [8]:
gdp_income_df = pd.read_csv(
    DATA_DIR / "GDP_Income_Data.csv",
    usecols=["GeoFIPS", "GeoName", "Description", "2022"], 
    skiprows=[0, 1, 2, 212, 213, 214, 215, 216, 217]
)
gdp_income_df = gdp_income_df.dropna()
gdp_income_df = gdp_income_df[gdp_income_df["GeoName"] != "United States"]
gdp_income_df["GeoName"] = gdp_income_df["GeoName"].replace(
    "District of Columbia", "Washington DC"
)
gdp_income_df.insert(
    gdp_income_df.columns.get_loc("GeoName") + 1,
    "StateCode",
    gdp_income_df["GeoName"].map(state_to_code),
)
gdp_income_df

,GeoFIPS,GeoName,StateCode,Description,2022
5,1000,Alabama,1,Real GDP (millions of chained 2017 dollars) 1,240210.5
7,1000,Alabama,1,Real per capita personal income 4,50649.0
9,2000,Alaska,2,Real GDP (millions of chained 2017 dollars) 1,50384.5
11,2000,Alaska,2,Real per capita personal income 4,58413.0
13,4000,Arizona,4,Real GDP (millions of chained 2017 dollars) 1,414460.2
...,...,...,...,...,...
199,54000,West Virginia,54,Real per capita personal income 4,48517.0
201,55000,Wisconsin,55,Real GDP (millions of chained 2017 dollars) 1,342786.9
203,55000,Wisconsin,55,Real per capita personal income 4,57706.0
205,56000,Wyoming,56,Real GDP (millions of chained 2017 dollars) 1,37338.1


## Load in census data

In [9]:
census_2022_df = pd.read_excel(
    DATA_DIR / "NST-EST2022-POP.xlsx",
    skiprows=[0, 1, 2, 3, 4, 5, 6, 7, 60, 61, 62, 63, 64, 65, 66]
)
census_2022_df.columns = ["State", "Base", "2020", "2021", "2022"]
census_2022_df["State"] = census_2022_df["State"].str.lstrip(".")
census_2022_df["State"] = census_2022_df["State"].replace(
    "District of Columbia", "Washington DC"
)
census_2022_df.insert(
    census_2022_df.columns.get_loc("State") + 1,
    "StateCode",
    census_2022_df["State"].map(state_to_code),
)
census_2022_df = census_2022_df.drop(["Base", "2020", "2021"], axis=1)
census_2022_df

,State,StateCode,2022
0,Alabama,1,5074296
1,Alaska,2,733583
2,Arizona,4,7359197
3,Arkansas,5,3045637
4,California,6,39029342
5,Colorado,8,5839926
6,Connecticut,9,3626205
7,Delaware,10,1018396
8,Washington DC,11,671803
9,Florida,12,22244823


## Construct datasets for forecasting $P_{ic}$ and $A_{jc}$

In [10]:
# Align everything on integer state codes, in the same order used for the OD matrices below
state_index = pd.Index(list(state_to_code.values()), name="StateCode")
E = E_prod.copy()
E.index = E.index.astype(int)
E = E.reindex(state_index)

def gdp_income_series(keyword):
    rows = gdp_income_df[gdp_income_df["Description"].str.contains(keyword, regex=False)]
    return rows.set_index("StateCode")["2022"].astype(float).reindex(state_index)

real_gdp = gdp_income_series("Real GDP")
real_income = gdp_income_series("Real per capita personal income")
population = census_2022_df.set_index("StateCode")["2022"].astype(float).reindex(state_index)

X_P = pd.DataFrame({
    "emp_agriculture": E["emp_agriculture"],
    "emp_manufacturing_mining": E["emp_manufacturing"] + E["emp_mining"],
    # "emp_wholesale_warehousing": E["emp_wholesale_warehousing"],
    "real_gdp": real_gdp,
}, index=state_index)

X_A = pd.DataFrame({
    "population": population,
    "real_per_capita_income": real_income,
    "emp_manufacturing": E["emp_manufacturing"],
}, index=state_index)

assert X_P.shape == (51, 3) and X_A.shape == (51, 3)
assert not X_P.isna().any().any() and not X_A.isna().any().any()

X_P

,emp_agriculture,emp_manufacturing_mining,real_gdp
StateCode,,,
1,5501.0,272560.0,240210.5
2,675.0,20038.0,50384.5
4,1941.0,183238.0,414460.2
5,4780.0,164315.0,137623.6
6,33361.0,1176838.0,3161153.4
8,1754.0,154455.0,422368.4
9,325.0,152378.0,280823.1
10,136.0,30038.0,85292.7
11,15.0,1014.0,145227.3


In [11]:
X_A

,population,real_per_capita_income,emp_manufacturing
StateCode,,,
1,5074296.0,50649.0,267658.0
2,733583.0,58413.0,10824.0
4,7359197.0,50935.0,171133.0
5,3045637.0,53291.0,161239.0
6,39029342.0,59126.0,1160856.0
8,5839926.0,64668.0,135443.0
9,3626205.0,69250.0,151851.0
10,1018396.0,56371.0,29767.0
11,671803.0,77241.0,1014.0


## Load in FAF6 data

In [12]:
df = pd.read_csv(DATA_DIR / "state" / "FAF6.0_State.csv")
# Filter to domestic flows
# TODO: Decide whether we want to do this
# df = df[df["trade_type"] == 1]
df

,fr_orig,dms_origst,dms_destst,fr_dest,fr_inmode,dms_mode,fr_outmode,sctg2,trade_type,tons_2022,value_2022
0,NaN,1,1,NaN,NaN,1,NaN,1,1,924.475048,1800.959032
1,NaN,1,5,NaN,NaN,1,NaN,1,1,88.641677,170.193456
2,NaN,1,12,NaN,NaN,1,NaN,1,1,28.002798,53.972010
3,NaN,1,13,NaN,NaN,1,NaN,1,1,150.320804,292.338206
4,NaN,1,20,NaN,NaN,1,NaN,1,1,9.563894,30.829675
...,...,...,...,...,...,...,...,...,...,...,...
599524,NaN,53,53,808.0,NaN,1,3.0,43,3,174.640924,706.122872
599525,NaN,53,53,808.0,NaN,1,4.0,43,3,0.000042,0.001840
599526,NaN,55,6,808.0,NaN,5,3.0,43,3,0.001996,0.007644
599527,NaN,55,45,808.0,NaN,2,3.0,43,3,0.000174,0.001099


## Create OD matrices for each commodity

In [13]:
state_codes = list(state_to_code.values())
commodity_codes = list(code_to_commodity.keys())

totals = (
    df.groupby(["sctg2", "dms_origst", "dms_destst"])["tons_2022"]
    .sum()
    .reindex(
        pd.MultiIndex.from_product(
            [commodity_codes, state_codes, state_codes],
            names=["sctg2", "dms_origst", "dms_destst"],
        ),
        fill_value=0,
    )
)

od_matrices = {
    code: totals.loc[code]
    .unstack("dms_destst")
    .reindex(index=state_codes, columns=state_codes)
    for code in commodity_codes
}

# This is the OD matrix corresponding to commodity "1" (Live animals/fish)
od_matrices[1]

dms_destst,1,2,4,5,6,8,9,10,11,12,...,46,47,48,49,50,51,53,54,55,56
dms_origst,,,,,,,,,,,,,,,,,,,,,
1,941.067092,0.000000,0.000000,88.641677,0.000000,0.000000,0.000000,0.000000,0.000000,28.031736,...,0.000000,53.476749,51.355797,0.000000,0.000000,0.157649,0.167137,0.000000,0.000000,0.000000
2,0.000000,2484.783180,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000033,...,0.000000,0.000000,0.004790,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.000000,0.000119,227.349691,0.000000,105.312465,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,1.892019,0.000000,0.000000,0.000000,0.033376,0.000000,0.000000,0.000000
5,135.295282,0.000000,0.000000,696.406032,0.000110,0.000000,0.000000,0.000000,0.000000,0.021470,...,0.000000,98.991316,116.064751,0.000000,0.000000,0.000000,0.256059,0.000000,0.000000,0.000000
6,0.000000,0.000139,19.537216,0.000000,4858.642969,0.020669,0.000000,0.000000,0.008203,0.032236,...,0.155812,0.092433,0.310786,0.000259,0.000000,0.000428,1.367144,0.000000,0.000038,0.001984
8,0.000000,0.000000,0.000000,0.000000,0.001653,1396.455311,0.000000,0.000000,0.000000,0.001655,...,0.000000,0.100635,0.028522,0.000163,0.000000,0.048265,0.002743,0.000000,0.000000,0.000163
9,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,23.185287,0.000000,0.000000,0.000000,...,0.000000,0.000570,0.013547,0.000000,0.000000,0.858562,0.000000,0.000000,0.000000,0.000000
10,0.000000,0.000000,0.000000,0.000000,0.000001,0.000000,0.157876,15.489616,0.000000,0.000250,...,0.000000,0.000000,0.000000,0.000000,0.000000,2.331477,0.000000,0.000000,0.000000,0.000000
11,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000212,0.000000,...,0.000000,0.000000,0.000362,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


The FAF6 data gives us an OD matrix for each commodity. That is, we have $T_{ijc}$, where $i$ is the origin state, $j$ is the destination state, and $c$ is the commodity type. This gives us the base number of "productions" and "attractions," i.e., $P_{ic} = \sum_j T_{ijc}$ and $A_{jc} = \sum_i T_{ijc}$. We can regress over these quantities for each state and commodity. For productions, we have
\begin{align*}
P_{ic} = \alpha_c + \beta_c x_{i1} + \gamma_c x_{i2} + \delta_c x_{i3}
\end{align*}
where $x_{i1}$ is the goods-producing employment (manufacturing and mining) in state $i$, $x_{i2}$ is the wholesale and warehousing employment in state $i$, and $x_{i3}$ is the real state GDP of state $i$. Likewise, for attractions we have
\begin{align*}
A_{jc} = \tilde{\alpha}_c + \tilde{\beta}_c \tilde{x}_{i1} + \tilde{\gamma}_c \tilde{x}_{i2} + \tilde{\delta}_c \tilde{x}_{i3}
\end{align*}
where $\tilde{x}_{i1}$ is the population of state $j$, $\tilde{x}_{i2}$ is the real per capita personal income in state $j$, and $\tilde{x}_{i3}$ is the manufacturing employment in state $j$. These are the more readily available quantities I was able to find from official government sources, so I decided to go with them, and I think it checks out. The data matrices $X_P$ and $X_A$ are already created in an earlier code cell, so all that is left to do is regress over productions and attractions for each state and commodity. Note that we can maybe do some transformations like log-log or inverse sinh and compare their $R^2$ values. 

In [14]:
# check multicollinearity for X_P
X = sm.add_constant(X_P.to_numpy())
vif = [variance_inflation_factor(X, i) for i in range(1, X.shape[1])]
print(dict(zip(X_P.columns, vif)))

{'emp_agriculture': np.float64(2.095950701305157), 'emp_manufacturing_mining': np.float64(3.533320097783333), 'real_gdp': np.float64(4.643200862624585)}


In [15]:
# check multicollinearity for X_A
X = sm.add_constant(X_A.to_numpy())
vif = [variance_inflation_factor(X, i) for i in range(1, X.shape[1])]
print(dict(zip(X_A.columns, vif)))

{'population': np.float64(4.400117544230832), 'real_per_capita_income': np.float64(1.0258830593981818), 'emp_manufacturing': np.float64(4.44840148687837)}


In [16]:
# select a subset of commodities we want to do analysis on--these were randomly chosen
commodity_codes_subset = [1, 21, 28, 31, 32, 35]
commodity_names_subset = [code_to_commodity[code] for code in commodity_codes_subset]

dict(zip(commodity_codes_subset, commodity_names_subset))

{1: 'Live animals/fish',
 21: 'Pharmaceuticals',
 28: 'Paper articles',
 31: 'Nonmetal min. prods.',
 32: 'Base metals',
 35: 'Electronics'}

In [17]:
od_matrix_1 = od_matrices[1]
P_i = od_matrix_1.sum(axis=1).to_numpy()
A_j = od_matrix_1.sum(axis=0).to_numpy()

mod = sm.OLS(np.log(P_i), sm.add_constant(np.log(X_P.to_numpy())))
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.715
Model:                            OLS   Adj. R-squared:                  0.697
Method:                 Least Squares   F-statistic:                     39.34
Date:                Wed, 18 Nov 2026   Prob (F-statistic):           7.22e-13
Time:                        13:01:24   Log-Likelihood:                -83.904
No. Observations:                  51   AIC:                             175.8
Df Residuals:                      47   BIC:                             183.5
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.5612      2.229     -0.252      0.8

In [18]:
mod = sm.OLS(np.log(A_j), sm.add_constant(np.log(X_A.to_numpy())))
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.515
Model:                            OLS   Adj. R-squared:                  0.484
Method:                 Least Squares   F-statistic:                     16.66
Date:                Wed, 18 Nov 2026   Prob (F-statistic):           1.65e-07
Time:                        13:01:24   Log-Likelihood:                -80.963
No. Observations:                  51   AIC:                             169.9
Df Residuals:                      47   BIC:                             177.7
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        -19.9096     19.065     -1.044      0.3